In [106]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import torch
import torch.nn as nn
import torch.nn.functional as F


model_name = "EleutherAI/pythia-410M"
device = "cuda:0"

tokenizer = AutoTokenizer.from_pretrained(model_name)

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    torch_dtype=torch.float32
).to(device)

Loading weights:   0%|          | 0/292 [00:00<?, ?it/s]

In [107]:
prompt = "The future of AI is"

inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

outputs = model.generate(
    **inputs,
    max_new_tokens=50
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


The future of AI is not in the hands of a few, but in the hands of the masses.

The future of AI is not in the hands of a few, but in the hands of the masses.

The future of AI is not in the hands


In [ ]:
print(model)

In [ ]:
model.gpt_neox.layers

In [ ]:
model.gpt_neox.layers

In [ ]:
for i in range(6, 10):
    print(f"Layer {i}")
    print(model.gpt_neox.layers[i].mlp)

In [108]:
import torch
import torch.nn.functional as F

def compute_routing_entropy(router_logits):
    """
    router_logits: Tensor of shape [batch_size, seq_len, num_experts]
    """
    # 1. Convert raw router outputs into a sum-to-1 probability distribution
    probs = F.softmax(router_logits, dim=-1)
    
    # 2. Apply the entropy formula: -sum(p * log(p))
    # We add 1e-10 to prevent taking the log of 0, which causes NaNs
    entropy = -torch.sum(probs * torch.log(probs + 1e-10), dim=-1)
    
    # 3. Average across all tokens in your batch
    return entropy.mean().item()


import numpy as np

def compute_expert_specialization(expert_indices, input_ids, num_experts, vocab_size):
    """
    expert_indices: Tensor of shape [total_tokens] containing chosen expert indices
    input_ids: Tensor of shape [total_tokens] containing corresponding token vocabulary IDs
    """
    # Initialize a frequency table [num_experts, vocab_size]
    token_counts = np.zeros((num_experts, vocab_size))
    
    # Map which tokens went to which expert
    for token_id, exp_id in zip(input_ids.cpu().numpy(), expert_indices.cpu().numpy()):
        token_counts[exp_id, token_id] += 1
        
    # Normalize frequencies to turn rows into vectors
    norms = np.linalg.norm(token_counts, axis=1, keepdims=True) + 1e-10
    normalized_counts = token_counts / norms
    
    # Compute the dot product to get pairwise cosine similarities
    similarity_matrix = np.dot(normalized_counts, normalized_counts.T)
    
    # Average the upper triangle (ignoring the diagonal self-similarity of 1.0)
    triu_indices = np.triu_indices(num_experts, k=1)
    avg_similarity = similarity_matrix[triu_indices].mean()
    
    return avg_similarity


def compute_perplexity(lm_loss):
    """
    lm_loss: Scalar cross-entropy loss output from your transformer model
    """
    loss_val = lm_loss.item() if hasattr(lm_loss, 'item') else lm_loss
    return np.exp(loss_val)

In [109]:
# ============================================================
#  MoE Upcycling — Comprehensive Metrics Tracker  (v2)
# ============================================================
import torch
import torch.nn.functional as F
import numpy as np
import math, time
from collections import defaultdict, Counter
from dataclasses import dataclass, field
from typing import Dict, List


# ─────────────────────────────────────────────────────────────
#  1. FLOP ESTIMATOR  (analytical)
# ─────────────────────────────────────────────────────────────
def estimate_flops(model, seq_len: int, batch_size: int = 1,
                   middle_layers=None, num_experts: int = 4,
                   top_k: int = 2) -> Dict:
    """
    Analytical FLOP estimate for one forward pass.
    Dense rule: 2 * params * seq_len.
    MoE layers replace full MLP with (top_k/num_experts) fraction + router linear.
    """
    cfg = model.config
    H   = cfg.hidden_size
    FF  = getattr(cfg, "intermediate_size", 4 * H)
    L   = cfg.num_hidden_layers
    V   = cfg.vocab_size
    T   = seq_len

    attn_flops    = 2 * T * (4 * H * H + 2 * T * H)  # QKV + scores + out
    dense_mlp     = 2 * T * 2 * H * FF                 # two linears in MLP

    dense_flops   = L * (attn_flops + dense_mlp) + 2 * T * H * V

    moe_n         = len(middle_layers) if middle_layers else 0
    dense_n       = L - moe_n
    router_flops  = 2 * T * H * num_experts            # gate linear
    active_mlp    = (top_k / num_experts) * dense_mlp

    moe_flops = (
        dense_n * (attn_flops + dense_mlp)
        + moe_n * (attn_flops + active_mlp + router_flops)
        + 2 * T * H * V
    )

    return {
        "dense_gflops"         : dense_flops / 1e9,
        "moe_gflops"           : moe_flops   / 1e9,
        "router_overhead_gflops": moe_n * router_flops / 1e9,
        "flop_reduction_pct"   : (1 - moe_flops / dense_flops) * 100,
        "efficiency_ratio"     : dense_flops / moe_flops,
    }


# ─────────────────────────────────────────────────────────────
#  2. STATELESS ROUTER METRIC FUNCTIONS
#     All accept logits: [B, T, num_experts]  (raw gate outputs)
# ─────────────────────────────────────────────────────────────
def router_entropy(logits: torch.Tensor) -> float:
    """Normalised entropy ∈ [0,1].  1 = perfect balance, 0 = total collapse."""
    probs       = F.softmax(logits.float(), dim=-1)
    raw         = -(probs * torch.log(probs + 1e-10)).sum(dim=-1).mean()
    return (raw / math.log(logits.shape[-1])).item()

def router_confidence(logits: torch.Tensor) -> float:
    """Mean max-probability.  High → router is very certain (may ignore experts)."""
    return F.softmax(logits.float(), dim=-1).max(dim=-1).values.mean().item()

def load_balance_cv(logits: torch.Tensor) -> float:
    """Coefficient of variation of mean expert gate probabilities.  0 = perfect balance."""
    loads = F.softmax(logits.float(), dim=-1).view(-1, logits.shape[-1]).mean(0)
    return (loads.std() / (loads.mean() + 1e-10)).item()

def dead_expert_fraction(logits: torch.Tensor, threshold: float = 0.01) -> float:
    """Fraction of experts with mean gate prob < threshold."""
    loads = F.softmax(logits.float(), dim=-1).view(-1, logits.shape[-1]).mean(0)
    return (loads < threshold).float().mean().item()

def expert_load_pct(logits: torch.Tensor) -> Dict:
    loads = F.softmax(logits.float(), dim=-1).view(-1, logits.shape[-1]).mean(0)
    return {f"E{i}": round(float(loads[i]) * 100, 1) for i in range(len(loads))}


# ─────────────────────────────────────────────────────────────
#  3. EXPERT WEIGHT DIVERGENCE
# ─────────────────────────────────────────────────────────────
def expert_weight_divergence(moe_layer) -> Dict:
    """
    Cosine distance between up-projection weight vectors of every expert pair.
    Starts near 0 (noise-perturbed copies), grows as experts specialise.
    """
    flat = []
    for exp in moe_layer.experts:
        w = exp.dense_h_to_4h.weight.detach().float().view(-1)
        flat.append(w / (w.norm() + 1e-10))

    n = len(flat)
    dists = [
        1.0 - torch.dot(flat[i], flat[j]).item()
        for i in range(n) for j in range(i + 1, n)
    ]
    return {
        "mean_cos_dist": round(float(np.mean(dists)),  4),
        "max_cos_dist" : round(float(np.max(dists)),   4),
        "min_cos_dist" : round(float(np.min(dists)),   4),
    }


# ─────────────────────────────────────────────────────────────
#  4. METRICS TRACKER
# ─────────────────────────────────────────────────────────────
@dataclass
class StepMetrics:
    step: int; lm_loss: float; perplexity: float
    aux_loss: float; total_loss: float; grad_norm: float
    loss_scale: float; tokens_per_sec: float
    entropy:      Dict[int, float] = field(default_factory=dict)
    confidence:   Dict[int, float] = field(default_factory=dict)
    load_cv:      Dict[int, float] = field(default_factory=dict)
    dead_frac:    Dict[int, float] = field(default_factory=dict)
    expert_loads: Dict[int, Dict]  = field(default_factory=dict)


class MoEMetricsTracker:
    """
    Drop-in tracker for the MoE upcycling pipeline.

    Key fixes vs v1:
    • _eval_specialization hook calls module.gate(x) directly — no recursion.
    • _eval_perplexity uses tokenizer.pad_token_id for correct label masking.
    • Probe-based per-step metrics never touch the live computation graph.
    """

    def __init__(self, model, tokenizer, middle_layers: List[int],
                 num_experts: int = 4, top_k: int = 2,
                 log_interval: int = 10, eval_interval: int = 50):
        self.model         = model
        self.tokenizer     = tokenizer
        self.middle_layers = middle_layers
        self.num_experts   = num_experts
        self.top_k         = top_k
        self.log_interval  = log_interval
        self.eval_interval = eval_interval
        self.history: List[StepMetrics] = []

        # Snapshot router gate weights at upcycling time to track drift
        self._init_router_weights = {
            i: model.gpt_neox.layers[i].mlp.router.gate.weight.detach().float().clone()
            for i in middle_layers
        }

        # Pre-compute FLOP estimates
        self.flops = estimate_flops(model, seq_len=512, middle_layers=middle_layers,
                                    num_experts=num_experts, top_k=top_k)

    # ── FLOP REPORT ──────────────────────────────────────────
    def print_flop_report(self):
        f = self.flops
        print("\n" + "="*62)
        print("  FLOP ANALYSIS (seq_len=512, batch=1)".center(62))
        print("="*62)
        print(f"  Dense baseline           : {f['dense_gflops']:>10.2f} GFLOPs")
        print(f"  MoE model                : {f['moe_gflops']:>10.2f} GFLOPs")
        print(f"  Router overhead          : {f['router_overhead_gflops']:>10.4f} GFLOPs")
        print(f"  FLOP reduction           : {f['flop_reduction_pct']:>10.2f} %")
        print(f"  Efficiency ratio (D/M)   : {f['efficiency_ratio']:>10.2f} x")
        print("="*62)

    # ── ROUTER DRIFT ─────────────────────────────────────────
    def router_drift(self) -> Dict[int, float]:
        out = {}
        for i in self.middle_layers:
            cur  = self.model.gpt_neox.layers[i].mlp.router.gate.weight.detach().float()
            init = self._init_router_weights[i]
            out[i] = (cur - init).norm().item()
        return out

    # ── PROBE LOGITS (no recursion risk) ─────────────────────
    def _probe_logits(self, moe):
        """
        Get router logits for a tiny random batch using moe.router.gate directly.
        This is safe to call inside or outside hooks.
        """
        dtype  = moe.router.gate.weight.dtype
        device = moe.router.gate.weight.device
        probe  = torch.randn(1, 16, moe.hidden_size, device=device, dtype=dtype)
        # Call the gate linear directly — NOT moe.router(probe) — to avoid
        # triggering any forward hooks that may be attached to the router module.
        with torch.no_grad():
            return moe.router.gate(probe.view(-1, moe.hidden_size))                       .view(1, 16, self.num_experts)

    # ── PER-STEP RECORD ──────────────────────────────────────
    def record_step(self, step, lm_loss, aux_loss, total_loss,
                    grad_norm, scaler, elapsed: float, batch_tokens: int) -> StepMetrics:
        lm  = lm_loss.item()  if hasattr(lm_loss,  "item") else float(lm_loss)
        aux = aux_loss.item() if hasattr(aux_loss, "item") else float(aux_loss)
        tot = total_loss.item() if hasattr(total_loss, "item") else float(total_loss)
        gn  = grad_norm.item() if hasattr(grad_norm, "item") else float(grad_norm)

        ent, conf, cv, dead, loads = {}, {}, {}, {}, {}
        for i in self.middle_layers:
            moe     = self.model.gpt_neox.layers[i].mlp
            logits  = self._probe_logits(moe)        # safe gate-direct call
            ent[i]  = router_entropy(logits)
            conf[i] = router_confidence(logits)
            cv[i]   = load_balance_cv(logits)
            dead[i] = dead_expert_fraction(logits)
            loads[i]= expert_load_pct(logits)

        m = StepMetrics(
            step=step, lm_loss=lm,
            perplexity=math.exp(min(lm, 20)),
            aux_loss=aux, total_loss=tot, grad_norm=gn,
            loss_scale=scaler.get_scale(),
            tokens_per_sec=batch_tokens / (elapsed + 1e-9),
            entropy=ent, confidence=conf, load_cv=cv,
            dead_frac=dead, expert_loads=loads,
        )
        self.history.append(m)
        return m

    def print_step(self, m: StepMetrics):
        print(f"\n[step {m.step:4d}]  lm={m.lm_loss:.4f} | ppl={m.perplexity:.2f} | "
              f"aux={m.aux_loss:.4f} | loss={m.total_loss:.4f} | "
              f"‖∇‖={m.grad_norm:.4f} | scale={m.loss_scale:.0f} | "
              f"tok/s={m.tokens_per_sec:.0f}")
        for i in self.middle_layers:
            loads_str = "  ".join(f"{k}:{v}%" for k, v in m.expert_loads[i].items())
            print(f"  └─ Lyr {i}: "
                  f"H={m.entropy[i]:.3f}  conf={m.confidence[i]:.3f}  "
                  f"CV={m.load_cv[i]:.3f}  dead={m.dead_frac[i]:.2f}  [{loads_str}]")

    # ── PERIODIC EVALUATION ───────────────────────────────────
    def evaluate(self, dataloader, max_batches: int = 5):
        print("\n" + "─"*62)
        print("  EVALUATION".center(62))
        print("─"*62)
        self._eval_perplexity(dataloader, max_batches)
        self._eval_specialization(dataloader, max_batches)
        self._eval_weight_divergence()
        drift = self.router_drift()
        print("\n  Router gate weight drift (L2 from init):")
        for i, d in drift.items():
            print(f"    Layer {i}: {d:.4f}")
        print("─"*62)

    def _eval_perplexity(self, dataloader, max_batches):
        """
        Validation perplexity.
        Uses tokenizer.pad_token_id (not hard-coded 0) for correct masking.
        """
        self.model.eval()
        pad_id = self.tokenizer.pad_token_id
        losses, device = [], next(self.model.parameters()).device
        with torch.no_grad():
            for n, batch in enumerate(dataloader):
                if n >= max_batches:
                    break
                ids  = batch["input_ids"].to(device)
                mask = batch["attention_mask"].to(device)
                # Only mask actual padding tokens
                lbl  = ids.clone()
                if pad_id is not None:
                    lbl[lbl == pad_id] = -100
                try:
                    out = self.model(input_ids=ids, attention_mask=mask, labels=lbl)
                    if not torch.isnan(out.loss) and not torch.isinf(out.loss):
                        losses.append(out.loss.item())
                except Exception:
                    continue
        self.model.train()
        if losses:
            avg = float(np.mean(losses))
            print(f"  Validation loss: {avg:.4f}  |  perplexity: {math.exp(min(avg, 20)):.2f}")
        else:
            print("  Validation: no valid batches")

    def _eval_specialization(self, dataloader, max_batches):
        """
        Expert token specialisation via vocabulary distribution overlap.

        FIX: the hook calls module.gate(x) directly — NOT module(x) —
        so it never re-enters the router's own forward() and cannot recurse.
        """
        vocab_size = self.tokenizer.vocab_size
        counts     = {i: np.zeros((self.num_experts, vocab_size))
                      for i in self.middle_layers}
        routing_buf= {i: None for i in self.middle_layers}
        hooks      = []

        def make_hook(layer_idx):
            def hook(module, inp, _out):
                # inp[0]: [B, T, H] hidden states passed into the router
                # Call .gate directly to get raw logits without triggering
                # this hook again (which would cause RecursionError).
                with torch.no_grad():
                    hidden = inp[0].float()          # ensure float32
                    flat   = hidden.view(-1, hidden.shape[-1])
                    logits = module.gate(flat)        # [B*T, num_experts]
                    routing_buf[layer_idx] = logits.argmax(dim=-1).cpu()
            return hook

        for i in self.middle_layers:
            h = self.model.gpt_neox.layers[i].mlp.router                     .register_forward_hook(make_hook(i))
            hooks.append(h)

        self.model.eval()
        device = next(self.model.parameters()).device
        with torch.no_grad():
            for n, batch in enumerate(dataloader):
                if n >= max_batches:
                    break
                ids = batch["input_ids"].to(device)
                try:
                    self.model(input_ids=ids)
                except Exception:
                    continue
                for i in self.middle_layers:
                    if routing_buf[i] is not None:
                        flat_ids = ids.cpu().numpy().flatten()
                        flat_exp = routing_buf[i].numpy().flatten()
                        for tok, exp in zip(flat_ids, flat_exp):
                            if tok < vocab_size:
                                counts[i][exp, tok] += 1

        for h in hooks:
            h.remove()
        self.model.train()

        print("\n  Expert specialisation (avg cosine sim; lower = more diverse experts):")
        for i in self.middle_layers:
            c   = counts[i]
            nrm = np.linalg.norm(c, axis=1, keepdims=True) + 1e-10
            sim = (c / nrm) @ (c / nrm).T
            idx = np.triu_indices(self.num_experts, k=1)
            avg = float(sim[idx].mean())
            top = {e: [self.tokenizer.decode([int(t)]).strip()
                        for t in np.argsort(c[e])[-3:][::-1]]
                   for e in range(self.num_experts)}
            print(f"    Layer {i}: avg_cos_sim={avg:.4f}  top tokens={top}")

    def _eval_weight_divergence(self):
        print("\n  Expert weight divergence (cosine distance; higher = more specialised):")
        for i in self.middle_layers:
            d = expert_weight_divergence(self.model.gpt_neox.layers[i].mlp)
            print(f"    Layer {i}: {d}")

    # ── END-OF-TRAINING SUMMARY ───────────────────────────────
    def print_summary(self):
        if not self.history:
            print("No steps recorded.")
            return
        ppls = [m.perplexity   for m in self.history]
        lms  = [m.lm_loss      for m in self.history]
        auxs = [m.aux_loss     for m in self.history]
        tps  = [m.tokens_per_sec for m in self.history]
        best_step = self.history[int(np.argmin(ppls))].step

        print("\n" + "="*62)
        print("  TRAINING SUMMARY".center(62))
        print("="*62)
        print(f"  Steps recorded : {len(self.history)}")
        print(f"  Initial PPL    : {ppls[0]:.2f}")
        print(f"  Final PPL      : {ppls[-1]:.2f}")
        print(f"  Best PPL       : {min(ppls):.2f}  at step {best_step}")
        print(f"  Avg aux loss   : {np.mean(auxs):.5f}")
        print(f"  Avg throughput : {np.mean(tps):.0f}  tok/s")
        f = self.history[-1]
        print("\n  Final router health:")
        for i in self.middle_layers:
            print(f"    Layer {i}: H={f.entropy[i]:.3f}  conf={f.confidence[i]:.3f}  "
                  f"CV={f.load_cv[i]:.3f}  dead={f.dead_frac[i]:.2f}")
        self.print_flop_report()


In [ ]:
entropy_dict = {}

In [110]:
import copy
import torch
import torch.nn as nn
import torch.nn.functional as F
from typing import Dict



class MoEExpert(nn.Module):
    """Single expert: mirrors GPTNeoXMLP structure exactly."""

    def __init__(self, hidden_size: int, intermediate_size: int, act_layer: nn.Module):
        super().__init__()
        self.dense_h_to_4h = nn.Linear(hidden_size, intermediate_size, bias=True)
        self.dense_4h_to_h = nn.Linear(intermediate_size, hidden_size, bias=True)
        self.act = copy.deepcopy(act_layer) # Preserves exact architecture activation

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dense_4h_to_h(self.act(self.dense_h_to_4h(x)))

    @classmethod
    def from_mlp(cls, mlp):
        hidden_size = mlp.dense_h_to_4h.in_features
        intermediate_size = mlp.dense_h_to_4h.out_features
    
        device = mlp.dense_h_to_4h.weight.device
        dtype  = mlp.dense_h_to_4h.weight.dtype
    
        expert = cls(
            hidden_size,
            intermediate_size,
            mlp.act
        ).to(device=device, dtype=dtype)
    
        noise_std = 1e-3  # tune this

        with torch.no_grad():
            expert.dense_h_to_4h.weight.copy_(mlp.dense_h_to_4h.weight)
            expert.dense_h_to_4h.weight.add_(
                torch.randn_like(expert.dense_h_to_4h.weight) * noise_std
            )
        
            expert.dense_h_to_4h.bias.copy_(mlp.dense_h_to_4h.bias)
            expert.dense_h_to_4h.bias.add_(
                torch.randn_like(expert.dense_h_to_4h.bias) * noise_std
            )
        
            expert.dense_4h_to_h.weight.copy_(mlp.dense_4h_to_h.weight)
            expert.dense_4h_to_h.weight.add_(
                torch.randn_like(expert.dense_4h_to_h.weight) * noise_std
            )
        
            expert.dense_4h_to_h.bias.copy_(mlp.dense_4h_to_h.bias)
            expert.dense_4h_to_h.bias.add_(
                torch.randn_like(expert.dense_4h_to_h.bias) * noise_std
            )
    
        return expert



class MoERouter(nn.Module):
    def __init__(self, hidden_size, num_experts):
        super().__init__()
        self.gate = nn.Linear(hidden_size, num_experts, bias=False)
        self.num_experts = num_experts

    def forward(self, x):
        # x shape: [batch_size, seq_len, hidden_size]
        logits = self.gate(x)
        probs = F.softmax(logits, dim=-1) # Shape: [batch_size, seq_len, num_experts]
        
        # ─── CALCULATE AUXILIARY LOSS ───
        # Flatten across batch and sequence dimensions to get total token counts
        flat_probs = probs.view(-1, self.num_experts) 
        mean_probs = flat_probs.mean(dim=0) # Average routing probability for each expert
        
        # Ideal scenario: every expert gets exactly (1 / num_experts) of the probability
        ideal_prob = 1.0 / self.num_experts
        
        # The auxiliary loss is the variance scaled up to keep it stable
        # We multiply by num_experts squared so the scale stays consistent regardless of expert count
        aux_loss = self.num_experts * torch.sum((mean_probs - ideal_prob) ** 2)
        
        # Store it as a protected attribute so the training loop can extract it
        self._aux_loss = aux_loss
        
        # Return the chosen expert indices for your routing logic
        return logits

class MoELayer(nn.Module):
    """Top-2 Mixture-of-Experts FFN drop-in for GPTNeoXMLP."""

    def __init__(self, mlp: nn.Module, num_experts: int = 8):
        super().__init__()
        hidden_size       = mlp.dense_h_to_4h.in_features
        device = mlp.dense_h_to_4h.weight.device
        dtype  = mlp.dense_h_to_4h.weight.dtype
 
        self.num_experts = num_experts
        self.hidden_size = hidden_size
 
        # Build experts – each starts as a copy of the original MLP
        self.experts = nn.ModuleList(
            [MoEExpert.from_mlp(mlp) for _ in range(num_experts)]
        )
        self.router = MoERouter(hidden_size, num_experts).to(device=device, dtype=dtype)
        self.router_entropy = []
        self.seq_tracking = 0
        self.pass_tracking = 0
 
    def _route(self, x: torch.Tensor):
        B, T, H = x.shape
        flat_x = x.view(B * T, H)                                 

        probs       = self.router(x).view(B * T, -1)       
        expert_weight, expert_idx = probs.max(dim=-1)      
        return expert_idx, expert_weight, flat_x
 
    def forward(self, hidden_states):
        batch_size, seq_len, hidden_dim = hidden_states.shape

        print(f"Batch Size: {batch_size}, Seq Len: {seq_len}, Hidden Dim:{hidden_dim}")

        if self.seq_tracking == seq_len:
            pass_name = "pass"+str(pass_tracking)
            entropy_dict[pass_name] = self.router_entropy
            self.router_entropy = []
            self.seq_tracking = 0
            self.pass_tracking = self.pass_tracking + 1

        self.seq_tracking = self.seq_tracking + 1
        
        print(f"Seq Tracking:{self.seq_tracking}")
        
        # 1. Get raw scores from the router
        router_logits = self.router(hidden_states) 

        print("Router Entropy: ", compute_routing_entropy(router_logits))
        self.router_entropy.append(compute_routing_entropy(router_logits))
        
        # 2. Get the Top-2 experts and their raw logits
        # top2_logits and top2_indices will have shape: [batch_size, seq_len, 2]
        top2_logits, top2_indices = torch.topk(router_logits, k=2, dim=-1)
        
        # 3. Re-normalize the weights of the selected 2 experts using Softmax
        # so they sum up to 1.0 for each token.
        top2_weights = F.softmax(top2_logits, dim=-1)
        
        # 4. Flatten the batch and seq_len dimensions to make routing easier
        flat_hidden = hidden_states.view(-1, hidden_dim)
        flat_indices = top2_indices.view(-1, 2)
        flat_weights = top2_weights.view(-1, 2)
        
        # Create an empty tensor to store the final combined outputs
        flat_output = torch.zeros_like(flat_hidden)
        
        # 5. Route the tokens to the experts
        for expert_idx, expert in enumerate(self.experts):
            # Find which tokens selected THIS expert as their #1 choice
            mask1 = (flat_indices[:, 0] == expert_idx)
            # Find which tokens selected THIS expert as their #2 choice
            mask2 = (flat_indices[:, 1] == expert_idx)
            
            # If this expert was chosen as the #1 choice by any tokens:
            if mask1.any():
                expert_out = expert(flat_hidden[mask1])
                # Scale by the weight of the #1 choice and add to output
                weight1 = flat_weights[mask1, 0].unsqueeze(-1)
                flat_output[mask1] += expert_out * weight1
                
            # If this expert was chosen as the #2 choice by any tokens:
            if mask2.any():
                expert_out = expert(flat_hidden[mask2])
                # Scale by the weight of the #2 choice and add to output
                weight2 = flat_weights[mask2, 1].unsqueeze(-1)
                flat_output[mask2] += expert_out * weight2

        # 6. Reshape back to the original format [batch_size, seq_len, hidden_dim]
        final_output = flat_output.view(batch_size, seq_len, hidden_dim)
        
        # Note: If you are calculating auxiliary loss, you would also return 
        # the router_logits or top2_weights here.
        return final_output

In [74]:
# -----------------------------------------------------------------------
# Convenience helper
# -----------------------------------------------------------------------
 
def replace_mlp_with_moe(model: nn.Module, num_experts: int = 8) -> nn.Module:
    """
    Walk the model and swap every GPTNeoXMLP with a MoELayer in-place.
 
    Example:
        model = replace_mlp_with_moe(model, num_experts=8)
    """
    for layer in model.gpt_neox.layers:           # adjust path if needed
        layer.mlp = MoELayer(layer.mlp, num_experts=num_experts)
    return model
 

In [111]:
# Replaces layers 5, 6, and 7 with 4-expert MoEs
middle_layers = [6, 7, 8] 

for i in middle_layers:
    old_mlp = model.gpt_neox.layers[i].mlp
    
    # Initialization automatically maps to the correct device and dtype
    moe = MoELayer(old_mlp, num_experts=4)
    
    # Direct replacement in the Hugging Face module dictionary
    model.gpt_neox.layers[i].mlp = moe
    
    # Optional but recommended: delete the old mlp to free VRAM immediately
    del old_mlp
    torch.cuda.empty_cache()

In [76]:
for i in middle_layers:
    moe = model.gpt_neox.layers[i].mlp
    print(f"Layer {i}:")
    print(f"  router dtype : {moe.router.gate.weight.dtype}")
    print(f"  expert 0 dtype: {moe.experts[0].dense_h_to_4h.weight.dtype}")

Layer 6:
  router dtype : torch.float32
  expert 0 dtype: torch.float32
Layer 7:
  router dtype : torch.float32
  expert 0 dtype: torch.float32
Layer 8:
  router dtype : torch.float32
  expert 0 dtype: torch.float32


In [ ]:
print(model.gpt_neox.layers)

In [112]:
inputs = tokenizer(
    "Hello world",
    return_tensors="pt"
).to(model.device)

with torch.no_grad():
    outputs = model(**inputs)

print(outputs.logits.shape)

Batch Size: 1, Seq Len: 2, Hidden Dim:1024
Seq Tracking:1
Router Entropy:  1.3294883966445923
Batch Size: 1, Seq Len: 2, Hidden Dim:1024
Seq Tracking:1
Router Entropy:  1.3538856506347656
Batch Size: 1, Seq Len: 2, Hidden Dim:1024
Seq Tracking:1
Router Entropy:  1.2619314193725586
torch.Size([1, 2, 50304])


In [ ]:
entropy_dict = {}

In [113]:
prompt = "The future of AI is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=30
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


Batch Size: 1, Seq Len: 5, Hidden Dim:1024
Seq Tracking:2
Router Entropy:  1.2248035669326782
Batch Size: 1, Seq Len: 5, Hidden Dim:1024
Seq Tracking:2
Router Entropy:  1.3639017343521118
Batch Size: 1, Seq Len: 5, Hidden Dim:1024
Seq Tracking:2
Router Entropy:  1.3160814046859741
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:3
Router Entropy:  1.291452169418335
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:3
Router Entropy:  1.310058832168579
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:3
Router Entropy:  1.3587138652801514
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:4
Router Entropy:  1.287248969078064
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:4
Router Entropy:  1.3228914737701416
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:4
Router Entropy:  1.3150699138641357
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:5
Router Entropy:  1.2982611656188965
Batch Size: 1, Seq Len: 1, Hidden Dim:1024
Seq Tracking:5
Route

We just added MoE to all middle layers, and the result changed.

In [114]:
import torch
import torch.nn.functional as F
from torch.optim import AdamW
from transformers import get_cosine_schedule_with_warmup
from datasets import load_dataset
from torch.utils.data import DataLoader

# ── 1. Freeze everything except MoE layers ────────────────────────────────────
moe_layer_indices = set(middle_layers)

for name, param in model.named_parameters():
    # Parse layer index from param name, e.g. "gpt_neox.layers.7.mlp.experts..."
    parts = name.split(".")
    try:
        layer_idx = int(parts[2])  # layers.<idx>
        in_moe = layer_idx in moe_layer_indices
    except (IndexError, ValueError):
        in_moe = False

    # Freeze everything OUTSIDE moe layers; inside moe: freeze experts, train router
    if in_moe and "router" in name:
        param.requires_grad = True   # train the router
    elif in_moe and "experts" in name:
        param.requires_grad = True   # also train experts (upcycle fine-tune)
    else:
        param.requires_grad = False  # freeze base model

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total     = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.1f}%)")




Trainable: 100,737,024 / 480,889,856 (20.9%)


In [116]:
import torch
import torch.nn.functional as F
import types

# ─── 1. FIXED FORWARD PATCH ──────────────────────────────────────────────────
def moe_forward_with_aux(self, x: torch.Tensor) -> torch.Tensor:
    B, T, H = x.shape
    expert_idx, expert_weight, flat_x = self._route(x)

    logits = self.router(x)
    probs = F.softmax(logits, dim=-1).view(B * T, -1)
    self._aux_loss = load_balance_loss(probs, expert_idx, self.num_experts)

    # ─── THE FIX ─────────────────────────────────────────────────────────────
    # Force 'out' to match the active operational dtype (expert_weight.dtype).
    # This dynamically handles FP16/BF16 under autocast, or FP32 if autocast is off.
    out = torch.zeros_like(flat_x, dtype=expert_weight.dtype)
    
    for e_id, expert in enumerate(self.experts):
        mask = (expert_idx == e_id)
        if not mask.any():
            continue
        scale = expert_weight[mask].unsqueeze(-1)
        
        # Both sides are now perfectly matched (e.g., Half = Half)
        out[mask] = expert(flat_x[mask]) * scale

    return out.view(B, T, H)


# ─── 2. LOAD BALANCE LOSS (Kept your perfect logic) ─────────────────────────
def load_balance_loss(router_probs: torch.Tensor, expert_indices: torch.Tensor,
                      num_experts: int) -> torch.Tensor:
    """
    router_probs  : (N, E)  – softmax probabilities from the router
    expert_indices: (N,)    – chosen expert per token (argmax)
    """
    # Safe type casting to match precision context
    one_hot = F.one_hot(expert_indices, num_classes=num_experts).to(router_probs.dtype)
    
    tokens_per_expert = one_hot.mean(dim=0)
    router_prob_per_expert = router_probs.mean(dim=0)
    
    # Matches Switch Transformer formulation perfectly
    loss = num_experts * (tokens_per_expert * router_prob_per_expert).sum()
    return loss


# ─── 3. APPLY MONKEY PATCH ──────────────────────────────────────────────────
for i in middle_layers:
    moe = model.gpt_neox.layers[i].mlp
    moe.forward = types.MethodType(moe_forward_with_aux, moe)

In [ ]:
# # ── 3. Patch MoELayer.forward to return aux loss ─────────────────────────────
# # We monkey-patch forward to stash the aux loss on the module so the
# # training loop can collect it without touching the model's output signature.

# def moe_forward_with_aux(self, x: torch.Tensor) -> torch.Tensor:
#     # x = x.to(self.router.gate.weight.dtype)  # ← cast input to fp16
#     B, T, H = x.shape
#     expert_idx, expert_weight, flat_x = self._route(x)

#     probs = self.router(x).view(B * T, -1)
#     self._aux_loss = load_balance_loss(probs, expert_idx, self.num_experts)

#     out = torch.zeros_like(flat_x)
#     for e_id, expert in enumerate(self.experts):
#         mask = (expert_idx == e_id)
#         if not mask.any():
#             continue
#         scale = expert_weight[mask].unsqueeze(-1)
#         out[mask] = expert(flat_x[mask]) * scale

#     return out.view(B, T, H)

# # Apply patch to all MoE layers
# import types
# for i in middle_layers:
#     moe = model.gpt_neox.layers[i].mlp
#     moe.forward = types.MethodType(moe_forward_with_aux, moe)




In [117]:
# ── 4. Optimizer + scheduler ─────────────────────────────────────────────────
# Separate LRs: router needs a higher LR (it's random-init),
# experts need a smaller LR (they start from good pretrained weights).

router_params = [p for n, p in model.named_parameters()
                 if p.requires_grad and "router" in n]
expert_params = [p for n, p in model.named_parameters()
                 if p.requires_grad and "experts" in n]

optimizer = AdamW([
    {"params": router_params, "lr": 1e-4},   # higher: router is randomly init'd
    {"params": expert_params, "lr": 5e-5},   # lower:  experts start pretrained
], weight_decay=0.01)

NUM_STEPS   = 500
WARMUP      = 200
scheduler = get_cosine_schedule_with_warmup(optimizer, WARMUP, NUM_STEPS)




In [118]:
# ── 5. Training loop ──────────────────────────────────────────────────────────
AUX_WEIGHT = 0.005  # λ — scales aux loss relative to LM loss; start small

dataset = load_dataset("wikitext", "wikitext-103-raw-v1", split="train")


len(dataset)

1801350

In [119]:
dataset = dataset.select(range(10000))

In [120]:
dataset[10]

{'text': ' The game \'s battle system , the BliTZ system , is carried over directly from Valkyira Chronicles . During missions , players select each unit using a top @-@ down perspective of the battlefield map : once a character is selected , the player moves the character around the battlefield in third @-@ person . A character can only act once per @-@ turn , but characters can be granted multiple turns at the expense of other characters \' turns . Each character has a field and distance of movement limited by their Action Gauge . Up to nine characters can be assigned to a single mission . During gameplay , characters will call out if something happens to them , such as their health points ( HP ) getting low or being knocked out by enemy attacks . Each character has specific " Potentials " , skills unique to each character . They are divided into " Personal Potential " , which are innate skills that remain unaltered unless otherwise dictated by the story and can either help or impede

In [121]:
def raw_tokenize_function(examples):
    # Append EOS token to the end of every text segment so the model knows where boundaries are
    texts = [text + tokenizer.eos_token for text in examples["text"]]
    return tokenizer(texts, add_special_tokens=False)

# Map it across the dataset, dropping the raw text column
raw_tokenized = dataset.map(
    raw_tokenize_function, 
    batched=True, 
    remove_columns=dataset.column_names
)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [122]:
block_size = 512

def group_into_chunks(examples):
    # Concatenate all token lists into a single continuous stream
    concatenated_examples = {k: sum(examples[k], []) for k in examples.keys()}
    total_length = len(concatenated_examples[list(examples.keys())[0]])
    
    # Floor-divide to find how many full blocks we can make
    if total_length >= block_size:
        total_length = (total_length // block_size) * block_size
        
    # Split the continuous stream into blocks of exactly block_size
    result = {
        k: [t[i : i + block_size] for i in range(0, total_length, block_size)]
        for k, t in concatenated_examples.items()
    }
    return result

# This transforms your variable-length dataset into uniformly packed 512-token rows
packed_dataset = raw_tokenized.map(group_into_chunks, batched=True)

Map:   0%|          | 0/10000 [00:00<?, ? examples/s]

In [126]:
# Tell the dataset to output PyTorch tensors instead of Python lists
packed_dataset.set_format(type="torch")

# Now create your loader
loader = DataLoader(
    packed_dataset, 
    batch_size=4, 
    shuffle=True, 
)

In [100]:
tokenized_datasets

Dataset({
    features: ['input_ids', 'attention_mask'],
    num_rows: 10000
})

In [127]:
# ============================================================
#  Instrumented Training Loop
# ============================================================
from torch.cuda.amp import autocast, GradScaler
import time

scaler = GradScaler()
model.train()
step, skipped = 0, 0

tracker = MoEMetricsTracker(
    model          = model,
    tokenizer      = tokenizer,
    middle_layers  = middle_layers,
    num_experts    = 4,
    top_k          = 2,
    log_interval   = 10,
    eval_interval  = 50,
)
tracker.print_flop_report()

for batch in loader:
    if step >= NUM_STEPS:
        break

    input_ids      = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)

    if attention_mask.sum() == 0:
        skipped += 1; continue

    labels = input_ids.clone()
    pad_id = tokenizer.pad_token_id
    if pad_id is not None:
        labels[labels == pad_id] = -100          # ← correct masking
    if (labels != -100).sum() == 0:
        skipped += 1; continue

    t0 = time.perf_counter()
    try:
        with autocast(dtype=torch.float16):
            outputs  = model(input_ids=input_ids,
                             attention_mask=attention_mask,
                             labels=labels)
            lm_loss  = outputs.loss
            aux_losses = [
                moe._aux_loss
                for i in middle_layers
                if hasattr((moe := model.gpt_neox.layers[i].mlp), "_aux_loss")
                and not (torch.isnan(moe._aux_loss) or torch.isinf(moe._aux_loss))
            ]
            aux_loss   = sum(aux_losses) if aux_losses else torch.tensor(0.0, device=device)
            total_loss = lm_loss + AUX_WEIGHT * aux_loss
    except Exception as e:
        print(f"[step {step}] FORWARD FAILED: {e}"); skipped += 1; continue

    if torch.isnan(total_loss) or torch.isinf(total_loss):
        skipped += 1; continue

    optimizer.zero_grad()
    scaler.scale(total_loss).backward()
    scaler.unscale_(optimizer)
    grad_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=0.5)
    scaler.step(optimizer)
    scaler.update()
    scheduler.step()

    elapsed      = time.perf_counter() - t0
    batch_tokens = int(attention_mask.sum().item())

    if step % tracker.log_interval == 0:
        m = tracker.record_step(step, lm_loss, aux_loss, total_loss,
                                grad_norm, scaler, elapsed, batch_tokens)
        tracker.print_step(m)

    if step % tracker.eval_interval == 0 and step > 0:
        tracker.evaluate(loader, max_batches=5)

    step += 1

print(f"\nDone — {step} steps, {skipped} skipped")
tracker.print_summary()


/tmp/ipykernel_58/639752297.py:7: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_58/639752297.py:41: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast(dtype=torch.float16):



              FLOP ANALYSIS (seq_len=512, batch=1)            
  Dense baseline           :     387.76 GFLOPs
  MoE model                :     374.88 GFLOPs
  Router overhead          :     0.0126 GFLOPs
  FLOP reduction           :       3.32 %
  Efficiency ratio (D/M)   :       1.03 x


/tmp/ipykernel_58/639752297.py:66: UserWarning: Detected call of `lr_scheduler.step()` before `optimizer.step()`. In PyTorch 1.1.0 and later, you should call them in the opposite order: `optimizer.step()` before `lr_scheduler.step()`.  Failure to do this will result in PyTorch skipping the first value of the learning rate schedule. See more details at https://pytorch.org/docs/stable/optim.html#how-to-adjust-learning-rate
  scheduler.step()



[step    0]  lm=4.1324 | ppl=62.33 | aux=3.4191 | loss=4.1495 | ‖∇‖=inf | scale=32768 | tok/s=6297
  └─ Lyr 6: H=0.925  conf=0.403  CV=0.119  dead=0.00  [E0:22.0%  E1:23.7%  E2:29.0%  E3:25.2%]
  └─ Lyr 7: H=0.895  conf=0.443  CV=0.125  dead=0.00  [E0:22.3%  E1:22.3%  E2:27.7%  E3:27.7%]
  └─ Lyr 8: H=0.933  conf=0.390  CV=0.085  dead=0.00  [E0:25.9%  E1:23.8%  E2:22.7%  E3:27.5%]

[step   10]  lm=3.5461 | ppl=34.68 | aux=3.4231 | loss=3.5632 | ‖∇‖=3.9818 | scale=32768 | tok/s=6408
  └─ Lyr 6: H=0.913  conf=0.416  CV=0.119  dead=0.00  [E0:28.7%  E1:21.7%  E2:23.8%  E3:25.9%]
  └─ Lyr 7: H=0.931  conf=0.409  CV=0.063  dead=0.00  [E0:23.2%  E1:26.3%  E2:24.1%  E3:26.4%]
  └─ Lyr 8: H=0.926  conf=0.413  CV=0.067  dead=0.00  [E0:26.5%  E1:23.4%  E2:23.7%  E3:26.4%]

[step   20]  lm=3.8555 | ppl=47.25 | aux=3.4601 | loss=3.8728 | ‖∇‖=4.6845 | scale=32768 | tok/s=6124
  └─ Lyr 6: H=0.939  conf=0.370  CV=0.060  dead=0.00  [E0:23.3%  E1:26.9%  E2:25.0%  E3:24.7%]
  └─ Lyr 7: H=0.938  conf=0.3

In [ ]:
prompt = "The future of AI is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=30
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

In [ ]:
model.eval()

prompt = "The future of AI is"

inputs = tokenizer(
    prompt,
    return_tensors="pt"
).to(model.device)

with torch.no_grad():

    outputs = model.generate(
        **inputs,
        max_new_tokens=30
    )

print(
    tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
)

In [ ]:
model.eval()

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate("The theory of relativity states that"))
print(generate("The future of AI is"))

In [ ]:
model.eval()

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate("The theory of relativity states that"))
print(generate("The future of AI is"))

In [129]:
model.eval()

def generate(prompt, max_new_tokens=100):
    inputs = tokenizer(prompt, return_tensors="pt").to(device)
    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=True,
            temperature=0.7,
            top_p=0.9,
        )
    return tokenizer.decode(output[0], skip_special_tokens=True)

print(generate("The theory of relativity states that"))
print(generate("The future of AI is"))

Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.
Setting `pad_token_id` to `eos_token_id`:0 for open-end generation.


The theory of relativity states that a particle of mass m and charge q must be in the same state as a charged particle of mass m and charge q and that it is a superposition of these states. Thus, a particle of mass m and charge q is a particle of mass m and charge q and a particle of mass m and charge q is a particle of mass m and charge q. The particle of mass m and charge q is called a " positive particle " and the particle of mass m and charge q is called a "
The future of AI is still in flux, but it is clear that the promise of AI will not be fully realized until the next decade.



In [128]:
import os

SAVE_DIR = "./pythia-1b-moe_v5_sequence_packing"
os.makedirs(SAVE_DIR, exist_ok=True)

# ── Save ──────────────────────────────────────────────────────────────────────
# Save the full model state dict (includes MoE layers with their custom structure)
torch.save(model.state_dict(), f"{SAVE_DIR}/model.pt")

# Save tokenizer and config so you can reload cleanly
tokenizer.save_pretrained(SAVE_DIR)
model.config.save_pretrained(SAVE_DIR)

# Optionally save just the MoE layer states separately (useful for ablations)
moe_states = {
    i: model.gpt_neox.layers[i].mlp.state_dict()
    for i in middle_layers
}
torch.save(moe_states, f"{SAVE_DIR}/moe_layers.pt")

print(f"Saved to {SAVE_DIR}/")

Saved to ./pythia-1b-moe_v5_sequence_packing/


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from collections import Counter
import tabulate  # Optional: pip install tabulate for pretty printing, or use standard formatting

# ─── 1. SETUP & CONFIGURATION ──────────────────────────────────────────
# (Assuming your model, tokenizer, and middle_layers are already in memory)
model.eval()
device = next(model.parameters()).device

prompt = "The gravitational attraction between clouds of primordial hydrogen and clumps of dark matter in the early universe caused the hydrogen gas to coalesce, eventually condensing and fusing to form stars. At larger scales this resulted in galaxies and clusters, so gravity is a primary driver for the large-scale structures in the universe. Gravity has an infinite range, although its effects become weaker as objects get farther "
inputs = tokenizer(prompt, return_tensors="pt").to(device)
input_ids = inputs["input_ids"][0]
tokens = [tokenizer.decode([tok_id]) for tok_id in input_ids]

# Container to dynamically store routing decisions
# Structure: { layer_idx: [tensor_of_chosen_experts_per_step, ...] }
routing_tracker = {i: [] for i in middle_layers}
hooks = []

# ─── 2. DEFINE AND REGISTER HOOKS ──────────────────────────────────────
def make_routing_hook(layer_idx):
    def hook(module, input, output):
        """
        Interceptors the router's output during the forward pass.
        Assumes router output shape is (batch, seq_len, num_experts)
        """
        with torch.no_grad():
            # Get the argmax (chosen expert index) along the final dimension
            chosen_experts = output.argmax(dim=-1).detach().cpu()
            # Append decisions (flattened to handle both prompt processing and token generation)
            routing_tracker[layer_idx].append(chosen_experts.view(-1))
    return hook

# Attach hooks to the routers
for i in middle_layers:
    router_module = model.gpt_neox.layers[i].mlp.router
    hook_handle = router_module.register_forward_hook(make_routing_hook(i))
    hooks.append(hook_handle)

# ─── 3. RUN INFERENCE (GENERATION) ─────────────────────────────────────
print(f"Generating text for prompt: '{prompt}'...\n")

with torch.no_grad():
    output_ids = model.generate(
        **inputs,
        max_new_tokens=20,
        do_sample=False,        # Greedy decoding for deterministic tracking
        pad_token_id=tokenizer.eos_token_id
    )

# Clean up hooks immediately so they don't impact subsequent training/evals
for hook in hooks:
    hook.remove()

# Decode the fully generated sequence of tokens
all_generated_ids = output_ids[0]
all_tokens = [tokenizer.decode([tok_id]) for tok_id in all_generated_ids]

# ─── 4. AGGREGATE AND PROCESS ROUTING DATA ─────────────────────────────
# Concatenate all steps (prompt prefill pass + token-by-token generation passes)
final_routing = {}
for i in middle_layers:
    final_routing[i] = torch.cat(routing_tracker[i]).tolist()

num_experts = model.gpt_neox.layers[middle_layers[0]].mlp.num_experts

# ─── 5. VISUALIZATION 1: GLOBAL EXPERT LOADS ────────────────────────────
print("=" * 60)
print(" GLOBAL TOKEN DISTRIBUTION PER LAYER ".center(60, "#"))
print("=" * 60)

for i in middle_layers:
    layer_decisions = final_routing[i]
    total_tokens_processed = len(layer_decisions)
    counts = Counter(layer_decisions)
    
    print(f"\n[Layer {i}] Total Tokens Processed: {total_tokens_processed}")
    row_str = "  "
    for exp in range(num_experts):
        share = (counts[exp] / total_tokens_processed) * 100
        row_str += f"Expert {exp}: {share:5.1f}% ({counts[exp]:3d} tokens) | "
    print(row_str[:-3])

# ─── 6. VISUALIZATION 2: TOKEN-BY-TOKEN ROUTING ALIGNMENT ──────────────
print("\n" + "=" * 60)
print(" TOKEN-BY-TOKEN ROUTING TRACE (First 15 Tokens) ".center(60, "#"))
print("=" * 60)

# Note: In autoregressive generation, the prompt tokens are evaluated all at once, 
# while generated tokens are calculated one-by-one. 
# The accumulated routing array maps 1:1 with our sequence length.

header = f"{'Token':<18} | " + " | ".join(f"Lyr {i}" for i in middle_layers)
print(header)
print("-" * len(header))

# Display up to the length of tracked tokens to avoid any sequence mismatch edge cases
display_limit = min(len(all_tokens), len(final_routing[middle_layers[0]]))

for t_idx in range(display_limit):
    # Escape newlines for cleaner console printing
    clean_token = all_tokens[t_idx].replace("\n", "\\n")
    
    layer_choices = []
    for i in middle_layers:
        expert_id = final_routing[i][t_idx]
        layer_choices.append(f" E{expert_id} ")
        
    choices_str = "  |  ".join(layer_choices)
    print(f"{clean_token:<18} |  {choices_str}")

In [ ]:
# Check if model object still has parameters at all
param_list = list(model.named_parameters())
print(f"Total params in model object: {len(param_list)}")
if param_list:
    name, p = param_list[0]
    print(f"First param: {name}, device: {p.device}, dtype: {p.dtype}")

In [ ]:
import numpy as np
import torch
import torch.nn.functional as F

def evaluate_expert_specialization(model, dataloader, tokenizer, upcycled_layers=[6, 7, 8], num_experts=4, max_batches=10):
    """
    Computes quantitative token clustering (expert specialization) via vocabulary distribution overlap.
    Safely recalculates router logits to avoid conflicts with custom monkey-patched layer outputs.
    """
    model.eval()
    vocab_size = tokenizer.vocab_size
    
    # Frequency table per layer: [num_experts, vocab_size]
    layer_token_counts = {
        i: np.zeros((num_experts, vocab_size)) for i in upcycled_layers
    }
    
    current_batch_routing = {i: None for i in upcycled_layers}
    hooks = []
    
    # SAFE HOOK FACTORY
    def make_spec_hook(layer_idx):
        def hook(module, input_tensor, output_tensor):
            with torch.no_grad():
                hidden_states = input_tensor[0].float()
                flat = hidden_states.view(-1, hidden_states.shape[-1])
                logits = module.gate(flat)          # ← call gate directly, no recursion
                expert_idx = logits.argmax(dim=-1).detach().cpu()
                current_batch_routing[layer_idx] = expert_idx
            return None
    return hook

    # Register hooks explicitly on the MoERouter modules
    for i in upcycled_layers:
        router_module = model.gpt_neox.layers[i].mlp.router
        hooks.append(router_module.register_forward_hook(make_spec_hook(i)))
        
    print(f"Evaluating expert specialization across {max_batches} validation batches...")
    
    batches_processed = 0
    with torch.no_grad():
        for batch in dataloader:
            if batches_processed >= max_batches:
                break
                
            input_ids = batch["input_ids"].to(model.device)
            
            # Forward pass triggers our registered hooks safely
            _ = model(input_ids=input_ids)
            
            # Populate frequency matrices
            for i in upcycled_layers:
                expert_idx = current_batch_routing[i]
                if expert_idx is None:
                    continue
                
                flat_input_ids = input_ids.cpu().numpy().flatten()
                flat_expert_idx = expert_idx.numpy().flatten()
                
                for token_id, exp_id in zip(flat_input_ids, flat_expert_idx):
                    if token_id < vocab_size:
                        layer_token_counts[i][exp_id, token_id] += 1
                        
            batches_processed += 1
            
    # Clean up hooks immediately
    for h in hooks:
        h.remove()
        
    print("\n================ EXPERT SPECIALIZATION REPORT ================")
    for i in upcycled_layers:
        counts = layer_token_counts[i]
        
        # Avoid division by zero if an expert hasn't processed any tokens yet
        norms = np.linalg.norm(counts, axis=1, keepdims=True) + 1e-10
        normalized_counts = counts / norms
        similarity_matrix = np.dot(normalized_counts, normalized_counts.T)
        
        triu_indices = np.triu_indices(num_experts, k=1)
        avg_similarity = similarity_matrix[triu_indices].mean()
        
        print(f"\n🔹 Layer {i} Overlap (Cosine Similarity): {avg_similarity:.4f}")
        print("  ┌─ Top Tokens per Expert:")
        
        for e_id in range(num_experts):
            top_token_ids = np.argsort(counts[e_id])[-5:][::-1]
            top_tokens = [tokenizer.decode([int(tid)]) for tid in top_token_ids]
            cleaned_tokens = [t.replace("\n", "\\n").replace(" ", "·") for t in top_tokens]
            print(f"  │  Expert {e_id}: {cleaned_tokens}")
            
    print("==============================================================")
    # model.train()

In [ ]:
evaluate_expert_specialization(
        model=model,
        dataloader=loader, # Uses your dataset loader
        tokenizer=tokenizer,
        upcycled_layers=middle_layers, # [6, 7, 8]
        num_experts=4,
        max_batches=5 # Keeps the evaluation quick
    )

In [130]:
# ============================================================
#  Post-Training Metrics Dashboard
#  Run this after training (or after loading a saved model)
# ============================================================

model.eval()

# ── 1. FLOP report ──
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
flops = estimate_flops(model, seq_len=512, middle_layers=middle_layers,
                       num_experts=4, top_k=2)
for k, v in flops.items():
    print(f"  {k:<30}: {v:.4f}")

# ── 2. Perplexity on a quick validation slice ──
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  VALIDATION PERPLEXITY")
device = next(model.parameters()).device
val_losses = []
with torch.no_grad():
    for i, batch in enumerate(loader):
        if i >= 20: break
        ids  = batch["input_ids"].to(device)
        mask = batch["attention_mask"].to(device)
        lbl = ids.clone()
        lbl[lbl == tokenizer.pad_token_id] = -100   # ← use actual pad id, not 0
        try:
            out = model(input_ids=ids, attention_mask=mask, labels=lbl)
            if not torch.isnan(out.loss) and not torch.isinf(out.loss):   # ← guard
                val_losses.append(out.loss.item())
        except Exception:
            continue
if val_losses:
    avg_loss = float(np.mean(val_losses))
    print(f"  avg loss = {avg_loss:.4f}  |  perplexity = {math.exp(min(avg_loss, 20)):.2f}")
else:
    print("  No valid batches — check pad_token_id and model state")

# ── 3. Router metrics across a validation batch ──
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  ROUTER HEALTH (per layer, random probe)")
for i in middle_layers:
    moe   = model.gpt_neox.layers[i].mlp
    probe = torch.randn(1, 32, moe.hidden_size,
                        device=device, dtype=moe.router.gate.weight.dtype)
    with torch.no_grad():
        # Call .gate directly — bypasses forward hooks, no recursion
        flat   = probe.view(-1, moe.hidden_size)
        logits = moe.router.gate(flat).view(1, 32, moe.num_experts)
    print(f"  Layer {i}:")
    print(f"    Normalised entropy   : {router_entropy(logits):.4f}  (1.0 = perfectly balanced)")
    print(f"    Router confidence    : {router_confidence(logits):.4f}  (lower = less certain)")
    print(f"    Load CV              : {load_balance_score(logits):.4f}  (0 = perfect balance)")
    print(f"    Dead expert fraction : {dead_expert_fraction(logits):.4f}  (0 = all experts active)")
    loads = expert_utilization_entropy(logits)
    print(f"    Expert loads         : {loads}")

# ── 4. Expert weight divergence ──
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
print("  EXPERT WEIGHT DIVERGENCE  (cosine distance, higher = more specialised)")
for i in middle_layers:
    moe = model.gpt_neox.layers[i].mlp
    div = compute_expert_weight_divergence(moe)
    print(f"  Layer {i}: {div}")

# ── 5. Full specialisation analysis ──
print("\n━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━")
evaluate_expert_specialization(
    model=model, dataloader=loader, tokenizer=tokenizer,
    upcycled_layers=middle_layers, num_experts=4, max_batches=10
)



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  dense_gflops                  : 387.7550
  moe_gflops                    : 374.8827
  router_overhead_gflops        : 0.0126
  flop_reduction_pct            : 3.3197
  efficiency_ratio              : 1.0343

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  VALIDATION PERPLEXITY
  avg loss = 2.6490  |  perplexity = 14.14

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
  ROUTER HEALTH (per layer, random probe)
  Layer 6:
    Normalised entropy   : 0.9295  (1.0 = perfectly balanced)
    Router confidence    : 0.4012  (lower = less certain)
    Load CV              : 0.1664  (0 = perfect balance)
    Dead expert fraction : 0.0000  (0 = all experts active)
    Expert loads         : {'expert_0_load': 0.20987989008426666, 'expert_1_load': 0.23517483472824097, 'expert_2_load': 0.3079240024089813, 'expert_3_load': 0.24702127277851105}
  Layer 7:
    Normalised entropy   : 0.9123  (1.0 = perfectly balanc